I'll start by installing required packages

In [15]:
pip install -q transformers datasets accelerate peft evaluate bitsandbytes sentencepiece

In [16]:
import torch, json, evaluate 
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [17]:
from huggingface_hub import login
login()

Quick check of token validity

In [18]:
from huggingface_hub import whoami
print(whoami()["auth"]["accessToken"]["role"])   # should print "read"

read


In [26]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,               # Store weights in 4-bit → 75 % VRAM saved
    bnb_4bit_quant_type="nf4",        # Normal-float-4: best quality for 4-bit
    bnb_4bit_compute_dtype=torch.float16,  # Mat-mul done in 16-bit for speed
)

model_name = "google/gemma-2b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)   
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [27]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 1,843,200 || all params: 2,508,015,616 || trainable%: 0.0735


In [40]:
from datasets import load_dataset

ds = load_dataset("qiaojin/PubMedQA", "pqa_artificial")
label2id = {"yes": 0, "no": 1, "maybe": 2}
id2label = {0: "yes", 1: "no", 2: "maybe"}

def fmt(batch):
    prompt = f"Question: {batch['question']}\nAnswer:"
    lbl_int = label2id[batch["final_decision"]]
    target = " " + id2label[lbl_int]
    return {"text": prompt + target}

train_val = ds["train"].train_test_split(test_size=0.15, seed=14)
train_ds = train_val["train"].select(range(20_000)).map(fmt, remove_columns=ds["train"].column_names)
val_ds = train_val["test"].map(fmt, remove_columns=ds["train"].column_names)


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/31691 [00:00<?, ? examples/s]

In [ ]:
max_len = 384          # short enough for T4, for answer should not troncate any, run a quick average and we are around 260 tokens 
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it", token=True)
tokenizer.pad_token = tokenizer.eos_token   # Gemma has no pad token

def tok_func(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_len,
        padding="max_length",   # fixed-length tensors
    )

train_tok = train_ds.map(tok_func, batched=True)
val_tok   = val_ds.map(tok_func, batched=True)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/31691 [00:00<?, ? examples/s]